In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import *
from gepkarva import gen, Cromosoma
from dataprocessor import datos_test_train

from sklearn.linear_model import LinearRegression

In [ ]:
# Cargar los datos
dfDiabetes = pd.read_csv("rawdata/Diabetes.csv")
dfCalifornia = pd.read_csv("rawdata/California.csv")

proporcion = 0.2
samples = 5
seed = 321938
rng = np.random.default_rng(seed)

DIABETES, CALIFORNIA = [], []
for _ in range(samples):
    dfD_train, dfD_test = datos_test_train(dfDiabetes, test_size=proporcion, rng=rng)
    dfC_train, dfC_test = datos_test_train(dfCalifornia, test_size=proporcion, rng=rng)

    colsD = dfDiabetes.columns
    X_Dtr = dfD_train[colsD[:-1]].to_numpy(); y_Dtr = dfD_train[colsD[-1]].to_numpy()
    X_Dte = dfD_test[colsD[:-1]].to_numpy(); y_Dte = dfD_test[colsD[-1]].to_numpy()

    colsC = dfCalifornia.columns
    X_Ctr = dfC_train[colsC[:-1]].to_numpy(); y_Ctr = dfC_train[colsC[-1]].to_numpy()
    X_Cte = dfC_test[colsC[:-1]].to_numpy(); y_Cte = dfC_test[colsC[-1]].to_numpy()

    DIABETES.append(((X_Dtr, y_Dtr), (X_Dte, y_Dte)))
    CALIFORNIA.append(((X_Ctr, y_Ctr), (X_Cte, y_Cte)))

# Precisión base de regresión lineal
---
1. **DIABETES**

In [3]:
rbase_D = {'Prueba': [], 'Score':[]}
for i in range(samples):
    train, test = DIABETES[i]
    reg_diabetes_base = LinearRegression().fit(train[0], train[1])
    score = reg_diabetes_base.score(test[0], test[1])
    rbase_D['Prueba'].append(i)
    rbase_D['Score'].append(score)    
rbase_D = pd.DataFrame(rbase_D)
rbase_D

,Prueba,Score
0,0,0.512839
1,1,0.364914
2,2,0.494556
3,3,0.406306
4,4,0.457187


2. **CALIFORNIA**

In [4]:
rbase_C = {'Prueba': [], 'Score':[]}
for i in range(samples):
    train, test = CALIFORNIA[i]
    reg_california_base = LinearRegression().fit(train[0], train[1])
    score = reg_california_base.score(test[0], test[1])
    rbase_C['Prueba'].append(i)
    rbase_C['Score'].append(score)    
rbase_C = pd.DataFrame(rbase_C)
rbase_C

,Prueba,Score
0,0,0.593635
1,1,0.603957
2,2,0.616563
3,3,0.612976
4,4,0.617671


# Evolucion de atributos

Vamos a intentar aprender 2 atributos primero y ver cómo afecta la regresión lineal. En funcióon de los resultados, observaremos si aprendiendo más atributos conseguimos algo o si deberíamos dirigir nuestros esfuerzos hacia otro lado.

In [5]:
# SETUP
size_poblacion0 = 100
h = 15
n_attr = 2
n_generaciones = 100

prob_mutacion = 0.1
prob_recombinacion = 0.3

In [6]:
def best_pobl(poblacion, train, p=90):
    r"""
    Parametros
    ----------
        - poblacion: lista de Cromosomas
        - X : matriz de datos (variables de los cromosomas)
        - y : target
    Returns
    -------
        - i : Indice del mejor individuo de la población
        - px : El percentil p de los mejores individuos
            Sobre los que podemos sesgar mutaciones y recombinaciones.
        - scores : lista de los scores de cada individuo.
    """
    scores = []
    X, y = train
    for c in poblacion:
        Xc = c.evaluar(X)
        datos = np.c_[X, Xc]
        regresion = LinearRegression().fit(datos, y)
        scores.append(regresion.score(datos, y))

    px = np.percentile(scores, p)
    i = np.argmax(scores)
    return i, px, scores

def test_cromosoma(c, regresion, test):
    X, y = test
    datos = np.c_[X, c.evaluar(X)]
    return regresion.score(datos, y)
    

1. **CALIFORNIA**

In [ ]:
n_vars = len(colsC)-1
R_Samples = []
c_plantilla = Cromosoma(n_vars=n_vars, h=h, n_gen=n_attr)
for i in range(samples):
    resultados = {'generacion': [], 'score': [], 'cromosoma': []}
    train, test = CALIFORNIA[i]
    poblacion0 = []
    for _ in range(size_poblacion0):
        c = Cromosoma(n_vars=n_vars, h=h, n_gen=n_attr)
        c.aleatorio(rng)
        poblacion0.append(c)

    i_best, p90, scores = best_pobl(poblacion0, train)
    resultados['generacion'].append(0)
    resultados['score'].append(scores[i_best])
    resultados['cromosoma'].append(poblacion0[i_best])

    for k in range(1, n_generaciones):
        resultados['generacion'].append(k)

        # Aplicamos operaciones genéticas sin competicion y
        # de manera aleatoria y sin elitismo

        # Recombinaciones simples
        recombinations = np.flatnonzero(rng.random(c_plantilla.largo_gen) < prob_recombinacion)
        pairs_recomb = rng.permutation(recombinations)
        for j in range(len(recombinations)):
            c1 = poblacion0[recombinations[j]]
            c2 = poblacion0[pairs_recomb[j]]
            recombinacion_simple(c1, c2, rng)

        mutations = np.flatnonzero(rng.random(c_plantilla.largo_gen) < prob_mutacion)
        for j in mutations:
            g = rng.integers(0,2)
            mutacion(poblacion0[j].sec_genes[g], prob_mutacion, rng)

        i_best, p90, scores = best_pobl(poblacion0, train)
        resultados['generacion'].append(k)
        resultados['score'].append(scores[i_best])
        resultados['cromosoma'].append(poblacion0[i_best])
    R_Samples.append(resultados)
        
    

In [9]:
d0 = R_Samples[0]
d1 = R_Samples[1]
d2 = R_Samples[2]
d3 = R_Samples[3]
d4 = R_Samples[4]

In [10]:
best0 = np.argmax(d0['score'])
best1 = np.argmax(d1['score'])
best2 = np.argmax(d2['score'])
best3 = np.argmax(d3['score'])
best4 = np.argmax(d4['score'])

In [34]:
Tr0, Te0 = CALIFORNIA[0]
c0 = d0['cromosoma'][best0]
Xtr0 = np.c_[Tr0[0], c0.evaluar(Tr0[0])]
Xte0 = np.c_[Te0[0], c0.evaluar(Te0[0])]
regresion0 = LinearRegression().fit(Xtr0, Tr0[1])
score = regresion0.score(Xte0, Te0[1])
c0.sec_genes[0].save_as_json('experimento1/best_sample0_g0')
c0.sec_genes[1].save_as_json('experimento1/best_sample0_g1')
print(score)

0.6780920136830961


In [35]:
Tr1, Te1 = CALIFORNIA[1]
c1 = d1['cromosoma'][best1]
Xtr1 = np.c_[Tr1[0], c0.evaluar(Tr1[0])]
Xte1 = np.c_[Te1[0], c0.evaluar(Te1[0])]
regresion1= LinearRegression().fit(Xtr1, Tr1[1])
score = regresion1.score(Xte1, Te1[1])
c1.sec_genes[0].save_as_json('experimento1/best_sample1_g0')
c1.sec_genes[1].save_as_json('experimento1/best_sample1_g1')
print(score)

0.6665198216820596


In [36]:
Tr2, Te2 = CALIFORNIA[2]
c2 = d2['cromosoma'][best2]
Xtr2 = np.c_[Tr2[0], c0.evaluar(Tr2[0])]
Xte2 = np.c_[Te2[0], c0.evaluar(Te2[0])]
regresion2= LinearRegression().fit(Xtr2, Tr2[1])
score = regresion2.score(Xte2, Te2[1])
c2.sec_genes[0].save_as_json('experimento1/best_sample2_g0')
c2.sec_genes[1].save_as_json('experimento1/best_sample2_g1')
print(score)

0.6624477650756031


In [37]:
Tr3, Te3 = CALIFORNIA[3]
c3 = d3['cromosoma'][best3]
Xtr3 = np.c_[Tr3[0], c0.evaluar(Tr3[0])]
Xte3 = np.c_[Te3[0], c0.evaluar(Te3[0])]
regresion3= LinearRegression().fit(Xtr3, Tr3[1])
score = regresion3.score(Xte3, Te3[1])
c3.sec_genes[0].save_as_json('experimento1/best_sample3_g0')
c3.sec_genes[1].save_as_json('experimento1/best_sample3_g1')
print(score)

0.6716810296247862


In [38]:
Tr4, Te4 = CALIFORNIA[4]
c4 = d4['cromosoma'][best4]
Xtr4 = np.c_[Tr4[0], c0.evaluar(Tr4[0])]
Xte4 = np.c_[Te4[0], c0.evaluar(Te4[0])]
regresion4= LinearRegression().fit(Xtr4, Tr4[1])
score = regresion4.score(Xte4, Te4[1])
c4.sec_genes[0].save_as_json('experimento1/best_sample4_g0')
c4.sec_genes[1].save_as_json('experimento1/best_sample4_g1')
print(score)

0.6766905270120743
